In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA
from src.benchmark import analytic_benchmark
from src.fully_connected import train_model
from src.helper import make_run_dir, save_run

## Load data

In [2]:
DATA_SET = 'rand_D'

df_train = pd.read_parquet(CLEAN_DATA / (DATA_SET + '_train.parquet'))
df_val   = pd.read_parquet(CLEAN_DATA / (DATA_SET + '_val.parquet'))
df_test  = pd.read_parquet(CLEAN_DATA / (DATA_SET + '_test.parquet'))


OUTPUT_DIR = make_run_dir()

## Hyperparameters & Features

In [3]:
BATCH_SIZE    = 4096
MAX_EPOCHS    = 100
PATIENCE      = 30
LR_PATIENCE   = 8
LR_FACTOR     = 0.3
INIT_LR       = 1e-3
ACTIVATION    = 'relu'
NEURONS        = 80
HIDDEN_LAYERS  = 3


FEATURES_3 = ['delta', 'T', 'spy_ret']
FEATURES_4 = ['delta', 'T', 'spy_ret', 'vix_lag']
TARGET      = 'd_iv'

## Analytic benchmark

In [4]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)

Analytic Benchmark
SSE = 8.3745  RMSE = 0.008338
Coefficients: a = -0.165665, b = -0.000516, c = 0.032673


## 3-Feature & 3-Feature ANN 

In [5]:
train_cfg = dict(
    epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, lr=INIT_LR,
    patience=PATIENCE, lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR,
    hidden_layers=HIDDEN_LAYERS, neurons=NEURONS, activation=ACTIVATION,
    target=TARGET,
)

result_3f = train_model(df_train, df_val, df_test,features=FEATURES_3, desc="ANN 3F", **train_cfg)

result_4f = train_model(df_train, df_val, df_test, features=FEATURES_4, desc="ANN 4F", **train_cfg)

ANN 3F:   0%|          | 0/100 epochs [00:00<?, ?epoch/s]  


Test:
SSE = 7.8642  RMSE = 0.008080  Time = 48.6s


ANN 4F:   0%|          | 0/100 epochs [00:00<?, ?epoch/s]  


Test:
SSE = 6.8504  RMSE = 0.007541  Time = 47.9s


In [6]:
summary = save_run(
    run_dir=OUTPUT_DIR,
    y_test=df_test[TARGET].values,
    hw=hw,
    models={"ANN-3F": result_3f, "ANN-4F": result_4f},
)
summary

,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic,Gain_Incremental
0,Analytic,8.374538,0.000070,0.008338,0.003523,0.000748,0.001885,0.125011,NaN,NaN,NaN
1,ANN-3F,7.864212,0.000065,0.008080,0.003401,0.000029,0.001782,0.178331,48.6s,6.09%,NaN
2,ANN-4F,6.850429,0.000057,0.007541,0.003275,0.000030,0.001845,0.284253,47.9s,18.20%,12.89%
3,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,96.5s,NaN,NaN
